# Assistant IA - Le Tropical (version fichiers PDF)

Deuxième version de l'assistant : au lieu d'un catalogue codé en dur, l'assistant s'appuie sur les **vrais documents PDF** de la marque, mélangeant produits et contenus hors-sujet. On y ajoute deux compétences par rapport au projet 1 :

- charger un dossier de fichiers PDF réels (document loader) ;
- découper ces documents en morceaux (chunking), car ils mêlent produits et articles ;
- prouver que la recherche sémantique ignore le bruit (sport, météo, actualité) présent dans les mêmes fichiers.

Le reste (rôle, outils, agent) est repris du projet 1. Embeddings multilingues locaux (gratuit, aucune clé supplémentaire).



In [1]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
ENV_FILE = Path(r"C:\Users\beruc\zando-na-ndako-api\.env")
load_dotenv(ENV_FILE)

model = ChatAnthropic(model="claude-sonnet-4-6")

C:\Users\beruc\AppData\Local\Temp\ipykernel_125796\3766046107.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


# 2 Charger les PDF

PyPDFDirectoryLoader lit tous les PDF du dossier data. Il renvoie un document par page, avec le fichier source et le numéro de page en métadonnées.

In [2]:
loader = PyPDFDirectoryLoader("data/")
documents = loader.load()

print(len(documents), "documents chargés")
print(documents[0].metadata)

3 documents chargés
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-10T21:05:45+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-10T21:05:45+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data\\letropical_cheveux_corps.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


# 3 Chunking

Chaque PDF mélange produits et articles hors-sujet. Si on vectorisait une page entière, le vecteur serait un mélange confus. Le chunking découpe le texte en petits morceaux, pour qu'un morceau parle d'une seule chose ; chaque morceau a alors son propre vecteur, net.

- chunk_size : taille maximale d'un morceau (trop grand mélange les sujets, trop petit coupe une fiche).
- chunk_overlap : les morceaux voisins partagent quelques caractères, pour ne pas couper une idée à la frontière.

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(documents)

print(len(chunks), "chunks créés")
print("---")
print(chunks[0].page_content)

22 chunks créés
---
Le TropicalSoins Cheveux et Corps
Masque capillaire au karité
Gamme Hair Tropical. Catégorie : Cheveux. Ingrédients : beurre de karité, huile d'avocat, huile de
coco. Usage : nourrit et revitalise les cheveux secs et abîmés. Prix : 16 €. Stock : 23.
Économie : marchés agricoles


# 4 Index sémantique (sur les chunks)

Identique au projet 1, mais on indexe les morceaux découpés.

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

print("Index prêt :", len(chunks), "chunks vectorisés")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index prêt : 22 chunks vectorisés


# 5 Test du bruit

On pose une question cosmétique. Les résultats doivent être des produits, et jamais un passage de sport, météo ou actualité, alors que ces articles vivent dans les mêmes PDF.

In [5]:
question = "je cherche un soin pour mes cheveux secs"
for d in vector_store.similarity_search(question, k=3):
    print(d.page_content)
    print("--- source:", d.metadata.get("source"), "\n")

Shampoing nourrissant au karité
Gamme Karité Hair. Catégorie : Cheveux. Ingrédients : beurre de karité, huile de coco, aloe vera.
Usage : nettoie et nourrit les cheveux secs. Prix : 13 €. Stock : 21.
Météo
--- source: data\letropical_cheveux_corps.pdf 

Après-shampoing réparateur
Gamme Hair Nature. Catégorie : Cheveux. Ingrédients : aloe vera, huile d'argan, beurre de karité.
Usage : facilite le démêlage et nourrit les longueurs. Prix : 13 €. Stock : 20.
Sport : basketball
--- source: data\letropical_cheveux_corps.pdf 

réduits pour les étudiants. La mise en service complète est prévue le mois prochain.
Huile à barbe
Gamme Barber Nature. Ingrédients : huiles de jojoba, argan et ricin, vitamine E. Usage : nourrit la
barbe et aide à l'assouplir. Prix : 15 €. Stock : 18.
Football : mercato
--- source: data\letropical_hommes.pdf 



# 6 Le rôle de l'assistant

Les instructions incluent une consigne anti-bruit : si un passage hors-sujet s'est glissé dans les résultats de recherche, l'assistant doit l'ignorer.

In [11]:
instructions = """Tu es l'assistant du service client de Le Tropical, une marque de cosmétiques congolaise en ligne. Tu es bilingue (français et anglais) et adoptes un ton chaleureux et professionnel.

TON RÔLE
Tu réponds aux questions sur les produits, les commandes et les livraisons.
Tu ne donnes jamais de conseil médical ni financier.
Tu ne réponds qu'aux sujets liés à Le Tropical ; tu n'inventes jamais d'information.
Les fiches produits proviennent de documents réels qui peuvent contenir, par erreur, des passages hors-sujet (sport, météo, actualité). Ignore-les : ne t'appuie que sur les informations cosmétiques.

CAS À TRANSMETTRE À UN CONSEILLER HUMAIN
1. Effet indésirable ou réaction (rougeur, allergie, irritation) : invite d'abord le client à consulter un médecin en cas de gêne persistante, puis préviens qu'un conseiller le recontactera sous 24h.
2. Produit défectueux ou colis endommagé : demande le numéro de commande, puis transmets à un conseiller qui recontactera sous 48h.
3. Réclamation sérieuse, litige ou client très mécontent : accuse réception avec empathie, puis transmets à un conseiller. (Pour un simple retard ou une question mineure, réponds toi-même.)
4. Question sur un produit que tu ne connais pas avec certitude : ne devine jamais ; indique que tu transmets la demande à un conseiller.

Quand tu transmets à un humain, donne toujours un délai et invite le client à écrire à beruchbordey@gmail.com en attendant."""

# 7 Les outils

chercher_produits est le RAG emballé en outil : il interroge l'index des PDF. Les autres outils (ticket, commande, stock) sont repris du projet 1. Les actions sont simulées.

In [12]:
@tool
def chercher_produits(question: str) -> str:
    """Recherche des produits dans le catalogue Le Tropical.
    À utiliser quand le client demande un conseil produit ou cherche un article."""
    resultats = vector_store.similarity_search(question, k=3)
    return "\n\n".join(d.page_content for d in resultats)


@tool
def creer_ticket_sav(sujet: str, details: str) -> str:
    """Crée un ticket au service client. À utiliser dès qu'un client signale
    une réclamation, un produit défectueux ou un effet indésirable."""
    return f"Ticket SAV créé — sujet : {sujet} | détails : {details}"


@tool
def verifier_commande(numero: str) -> str:
    """Vérifie le statut d'une commande à partir de son numéro.
    À utiliser quand un client demande où en est sa commande."""
    return f"Commande {numero} : expédiée, livraison prévue sous 3 jours."


@tool
def verifier_stock(produit: str) -> str:
    """Vérifie la disponibilité d'un produit dans le catalogue.
    À utiliser quand un client demande si un article est en stock."""
    return f"Le produit '{produit}' est actuellement en stock."


model_avec_outils = model.bind_tools(
    [chercher_produits, creer_ticket_sav, verifier_commande, verifier_stock]
)

# 8 La boucle agent

Mémoire (historique) + routage multi-outils (le dictionnaire outils) + boucle while pour enchaîner plusieurs outils si nécessaire. Le RAG des PDF est utilisé via l'outil chercher_produits.

In [13]:
def lancer_agent():
    historique = [SystemMessage(content=instructions)]
    outils = {
        "chercher_produits": chercher_produits,
        "creer_ticket_sav": creer_ticket_sav,
        "verifier_commande": verifier_commande,
        "verifier_stock": verifier_stock,
    }

    while True:
        question = input("Vous : ")
        if question.lower() in ["quit", "exit", "stop"]:
            break

        historique.append(HumanMessage(content=question))
        reponse = model_avec_outils.invoke(historique)
        historique.append(reponse)

        while reponse.tool_calls:
            for appel in reponse.tool_calls:
                outil = outils[appel["name"]]
                resultat = outil.invoke(appel["args"])
                historique.append(ToolMessage(content=resultat, tool_call_id=appel["id"]))
            reponse = model_avec_outils.invoke(historique)
            historique.append(reponse)

        print("Assistant :", reponse.content)

In [15]:

 lancer_agent()

Vous :  bonjour


Assistant : Bonjour ! 😊 Bienvenue chez **Le Tropical**, votre marque de cosmétiques congolaise !

Je suis ravi(e) de vous accueillir. Comment puis-je vous aider aujourd'hui ?

- 🛍️ **Conseil produit** – Vous cherchez un article en particulier ?
- 📦 **Suivi de commande** – Vous souhaitez savoir où en est votre livraison ?
- ❓ **Autre question** – N'hésitez pas à me poser toutes vos questions !

Je suis là pour vous ! 🌿


Vous :  stop


In [21]:
!git init

Reinitialized existing Git repository in C:/Users/beruc/Downloads/Projets defis 30 jours/projet langchain avec produit en pdf/.git/


In [23]:
!git add .gitignore

In [24]:
!git status

On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README_le_tropical_pdf_rag.md
	assistant_le_tropical_pdf_rag.ipynb
	data/



In [44]:
!git check-ignore -v .env

.gitignore:1:.env	.env


In [45]:
!git status --short

A  .gitignore
?? README_le_tropical_pdf_rag.md
?? assistant_le_tropical_pdf_rag.ipynb
?? data/


In [47]:
!type .gitignore

.env
__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
venv/


In [ ]:
!git add .

In [49]:
!git commit -m "mon premier premier projet langchain avec produit en pdf"

[master (root-commit) 243d27a] mon premier premier projet langchain avec produit en pdf
 1 file changed, 6 insertions(+)
 create mode 100644 .gitignore
